# Using `get_sunburst_figure_from_pivot` to create sunburst diagrams

This notebook demonstrates how to use the `get_sunburst_figure_from_pivot` function from `evbeantools.juptools` to create interactive [Plotly sunburst diagrams](https://plotly.com/python/sunburst-charts/) from hierarchical beancount data.

A sunburst diagram is a great way to visualize hierarchical financial data, such as expenses or income broken down by account categories and subcategories.

## Notebook preparation

In [1]:
import pandas as pd

from beancount.loader import load_string

from evbeantools.juptools import beanquery2df, get_bean_pivot
from evbeantools.juptools import get_sunburst_figure_from_pivot

In [2]:
pd.options.display.float_format = "{:,.2f}".format

## How `get_sunburst_figure_from_pivot` works

`get_sunburst_figure_from_pivot` takes two main inputs:

1. **`df`** — a pandas DataFrame with a **MultiIndex** representing the account hierarchy (e.g. `acc_L1`, `acc_L2`, ...) and one or more value columns.
2. **`column_to_pick`** — the name of the column (or a tuple for MultiIndex columns) containing the numeric values to display.

It also accepts an optional **`strict_mode`** parameter (default `True`):
- `True` — raises an error if any root-level account has a negative total value.
- `False` — silently excludes negative roots and their sub-trees.

The function returns a Plotly `Figure` with a sunburst chart, where:
- The hierarchy is derived from the DataFrame's MultiIndex.
- If a parent account has its own direct postings (in addition to child accounts), a placeholder node `parent_` is created to represent the parent's own value.

## Quick demo without a beancount file

Before loading a real beancount ledger, let's see how `get_sunburst_figure_from_pivot` works on a plain pandas DataFrame built from scratch. The key requirement is that **the DataFrame has a MultiIndex representing the hierarchy** and at least one numeric column to display.

### 1. Create sample data

We create a DataFrame with columns `acc_L1`, `acc_L2` (representing the account hierarchy) and `amount` (the numeric values). This is the structure that `get_sunburst_figure_from_pivot` expects: a **MultiIndex** built from account-level columns and a value column.

In [3]:
# Sample expense data — no beancount file needed
sample_data = pd.DataFrame({
    "acc_L1": [
        "Food", "Food", "Food",
        "Housing", "Housing", "Housing",
        "Transport", "Transport", "Transport",
        "Health", "Health",
        "Entertainment", "Entertainment", "Entertainment",
    ],
    "acc_L2": [
        "Groceries", "Restaurant", "Coffee",
        "Rent", "Utilities", "Insurance",
        "Fuel", "PublicTransport", "Maintenance",
        "Doctor", "Pharmacy",
        "Movies", "Books", "Sports",
    ],
    "amount": [
        4800, 3600, 1200,
        18000, 3600, 1200,
        2400, 960, 600,
        1200, 480,
        360, 240, 480,
    ],
})

sample_data

,acc_L1,acc_L2,amount
0,Food,Groceries,4800
1,Food,Restaurant,3600
2,Food,Coffee,1200
3,Housing,Rent,18000
4,Housing,Utilities,3600
5,Housing,Insurance,1200
6,Transport,Fuel,2400
7,Transport,PublicTransport,960
8,Transport,Maintenance,600
9,Health,Doctor,1200


### 2. Create a pivot table with standard pandas

We use `pd.pivot_table` to aggregate amounts by the hierarchy levels. The resulting DataFrame must have a **MultiIndex** built from the account-level columns (`acc_L1`, `acc_L2`).

In [4]:
sample_pivot = sample_data.pivot_table(
    index=["acc_L1", "acc_L2"],
    values="amount",
    aggfunc="sum",
)

sample_pivot

amount
acc_L1        acc_L2                 
Entertainment Books               240
              Movies              360
              Sports              480
Food          Coffee             1200
              Groceries          4800
              Restaurant         3600
Health        Doctor             1200
              Pharmacy            480
Housing       Insurance          1200
              Rent              18000
              Utilities          3600
Transport     Fuel               2400
              Maintenance         600
              PublicTransport     960

### 3. Generate the sunburst diagram

Pass the pivot DataFrame and the column name (`"amount"`) to `get_sunburst_figure_from_pivot`.

In [5]:
sample_fig = get_sunburst_figure_from_pivot(sample_pivot, column_to_pick="amount")

sample_fig.update_layout(
    margin=dict(t=30, l=0, r=0, b=0),
    width=650,
    height=650,
    title_text="Sample Expenses Sunburst",
    title_font=dict(size=20),
    font=dict(size=12),
)

sample_fig.show()

### Using `strict_mode=False` for data with negative values

By default, `get_sunburst_figure_from_pivot` operates in **strict mode**: it raises an error if any root-level node has a negative total. This is because sunburst diagrams require positive values.

If your data may contain negative root nodes (e.g. refunds that exceed expenses in a category), you can set `strict_mode=False` to silently exclude those branches instead of raising an error.

**Handling of negative values:**
- **Non-root negatives**: the entire sibling set (the negative node and its siblings) is removed, while the parent is retained.
- **Root negatives** (`strict_mode=False`): the root and its entire sub-tree are dropped.

In [6]:
# Example: a DataFrame with a negative root-level category
neg_data = pd.DataFrame({
    "acc_L1": ["Food", "Food", "Transport", "Refunds"],
    "acc_L2": ["Groceries", "Restaurant", "Fuel", "_"],
    "amount": [500.0, 300.0, 200.0, -50.0],
})

pivot_with_neg = neg_data.pivot_table(
    index=["acc_L1", "acc_L2"],
    values="amount",
    aggfunc="sum",
)

pivot_with_neg

amount
acc_L1    acc_L2            
Food      Groceries   500.00
          Restaurant  300.00
Refunds   _           -50.00
Transport Fuel        200.00

In [7]:
# strict_mode=False silently excludes the negative "Refunds" branch
fig_lenient = get_sunburst_figure_from_pivot(pivot_with_neg,
                                              column_to_pick="amount",
                                              strict_mode=False)

fig_lenient.update_layout(
    margin=dict(t=30, l=0, r=0, b=0),
    width=600,
    height=600,
    title_text="Expenses (negative branches excluded)",
    title_font=dict(size=20),
    font=dict(size=12),
)

fig_lenient.show()

## Using with a beancount file

Now let's see how to use `get_sunburst_figure_from_pivot` with real beancount data. We define a small demo ledger inline and load it with `load_string`.

The workflow is: load the ledger, query postings with `beanquery2df`, pivot with `get_bean_pivot`, and pass the result to `get_sunburst_figure_from_pivot`.

### Define and load a demo ledger

In [8]:
ledger = """\
option "operating_currency" "USD"

2023-01-01 open Assets:Bank:Checking
2023-01-01 open Expenses:Food:Groceries
2023-01-01 open Expenses:Food:Restaurant
2023-01-01 open Expenses:Food:Coffee
2023-01-01 open Expenses:Housing:Rent
2023-01-01 open Expenses:Housing:Utilities
2023-01-01 open Expenses:Transport:Fuel
2023-01-01 open Expenses:Transport:PublicTransport
2023-01-01 open Expenses:Health:Doctor
2023-01-01 open Expenses:Entertainment:Movies
2023-01-01 open Expenses:Entertainment:Books
2023-01-01 open Equity:Opening-Balances

2023-01-01 * "Opening Balance"
    Assets:Bank:Checking       50000 USD
    Equity:Opening-Balances

; ---- 2023 expenses ----
2023-02-15 * "Supermarket"
    Expenses:Food:Groceries     400 USD
    Assets:Bank:Checking

2023-03-10 * "Dinner out"
    Expenses:Food:Restaurant    120 USD
    Assets:Bank:Checking

2023-04-01 * "Monthly rent"
    Expenses:Housing:Rent      1500 USD
    Assets:Bank:Checking

2023-05-20 * "Electric bill"
    Expenses:Housing:Utilities  180 USD
    Assets:Bank:Checking

2023-06-15 * "Gas station"
    Expenses:Transport:Fuel     90 USD
    Assets:Bank:Checking

2023-07-10 * "Bus pass"
    Expenses:Transport:PublicTransport  50 USD
    Assets:Bank:Checking

2023-08-05 * "Annual checkup"
    Expenses:Health:Doctor      200 USD
    Assets:Bank:Checking

2023-09-22 * "Cinema"
    Expenses:Entertainment:Movies  30 USD
    Assets:Bank:Checking

2023-10-14 * "Bookstore"
    Expenses:Entertainment:Books   45 USD
    Assets:Bank:Checking

2023-11-01 * "Coffee shop"
    Expenses:Food:Coffee        60 USD
    Assets:Bank:Checking

; ---- 2024 expenses ----
2024-01-15 * "Supermarket"
    Expenses:Food:Groceries     450 USD
    Assets:Bank:Checking

2024-02-20 * "Dinner out"
    Expenses:Food:Restaurant    150 USD
    Assets:Bank:Checking

2024-03-01 * "Monthly rent"
    Expenses:Housing:Rent      1500 USD
    Assets:Bank:Checking

2024-04-18 * "Electric bill"
    Expenses:Housing:Utilities  200 USD
    Assets:Bank:Checking

2024-05-10 * "Gas station"
    Expenses:Transport:Fuel     110 USD
    Assets:Bank:Checking

2024-06-25 * "Annual checkup"
    Expenses:Health:Doctor      250 USD
    Assets:Bank:Checking

2024-07-30 * "Cinema"
    Expenses:Entertainment:Movies  35 USD
    Assets:Bank:Checking

2024-08-10 * "Coffee shop"
    Expenses:Food:Coffee        75 USD
    Assets:Bank:Checking
"""

entries, errors, options = load_string(ledger)

CURR = options["operating_currency"][0]
print(f"Operating currency: {CURR}")
print(f"Errors: {errors}")

Operating currency: USD
Errors: []


### Query expense postings

In [9]:
# Query all expense postings, converting to the operating currency
query = f"""
    SELECT id, date, account, CONVERT(position, '{CURR}', date) AS amount
    WHERE account ~ '^Expenses'
"""

expenses_df = beanquery2df(entries, options, query)
expenses_df.head(10)

,id,date,account,amount (USD)
0,0e9ac6344fd88fca2c9828c88154e143,2023-02-15,Expenses:Food:Groceries,400.00
1,a03389aae87182ec62e9a081a3b8e3a0,2023-03-10,Expenses:Food:Restaurant,120.00
2,17f795099f849992b390b48dfe04e409,2023-04-01,Expenses:Housing:Rent,"1,500.00"
3,cd9c62bca2dc0905d2f2ebfaef27577b,2023-05-20,Expenses:Housing:Utilities,180.00
4,1c89187972d4b95f345ccedaabb629ce,2023-06-15,Expenses:Transport:Fuel,90.00
5,344ecc110ba9b3ebf716001d626f708f,2023-07-10,Expenses:Transport:PublicTransport,50.00
6,271c3f8d1e96b25438e3a7f53405d92f,2023-08-05,Expenses:Health:Doctor,200.00
7,37ce55f7acd45201193c80071926c295,2023-09-22,Expenses:Entertainment:Movies,30.00
8,5094745c3f089f8b7dac40f065bcf4d8,2023-10-14,Expenses:Entertainment:Books,45.00
9,db8f49840d894522ec3f4d4574f1e42f,2023-11-01,Expenses:Food:Coffee,60.00


### Create a pivot table with `get_bean_pivot`

`get_bean_pivot` splits the `account` column into hierarchical levels (`acc_L1`, `acc_L2`, ...) and creates a pivot table. 

We assign a single-value `period` column to aggregate all time periods into one, which is the typical input for a sunburst diagram showing totals.

In [10]:
# We assign a single period value to combine all data into one column
expenses_df["period"] = "all_periods"

# Create the pivot table with full account hierarchy (max_row_levels=100)
expenses_pivot = get_bean_pivot(expenses_df,
                                column="period",
                                max_row_levels=100,
                                repeat_row_labels=False)

expenses_pivot

amount (USD)
period                         all_periods
acc_L1        acc_L2                      
Entertainment Books                  45.00
              Movies                 65.00
Food          Coffee                135.00
              Groceries             850.00
              Restaurant            270.00
Health        Doctor                450.00
Housing       Rent                3,000.00
              Utilities             380.00
Transport     Fuel                  200.00
              PublicTransport        50.00

### Select only the operating currency column

If the pivot table has MultiIndex columns (e.g. `(amount (USD), all_periods)`), we select only the column for the operating currency.

In [11]:
# Select only the operating currency values
expenses_pivot_curr = expenses_pivot.loc[:, f"amount ({CURR})"]
expenses_pivot_curr

period                         all_periods
acc_L1        acc_L2                      
Entertainment Books                  45.00
              Movies                 65.00
Food          Coffee                135.00
              Groceries             850.00
              Restaurant            270.00
Health        Doctor                450.00
Housing       Rent                3,000.00
              Utilities             380.00
Transport     Fuel                  200.00
              PublicTransport        50.00

## Step 2: Create the sunburst diagram

Now we pass the prepared DataFrame and the column name to `get_sunburst_figure_from_pivot`. The `column_to_pick` parameter specifies which column contains the values. In our case it is `"all_periods"` — the single period we assigned earlier.

In [12]:
fig = get_sunburst_figure_from_pivot(expenses_pivot_curr, column_to_pick="all_periods")

fig.update_layout(
    margin=dict(t=30, l=0, r=0, b=0),
    width=700,
    height=700,
    title_text="Expenses Sunburst",
    title_font=dict(size=20),
    font=dict(size=12),
)

fig.show()

## Controlling hierarchy depth with `max_row_levels`

By adjusting the `max_row_levels` parameter in `get_bean_pivot`, you control how many levels of account hierarchy appear in the sunburst. Setting `max_row_levels=1` shows only the top-level expense categories.

In [13]:
# Shallow pivot — only top-level expense categories
expenses_pivot_l1 = get_bean_pivot(expenses_df,
                                    column="period",
                                    max_row_levels=1,
                                    repeat_row_labels=False)

expenses_pivot_l1_curr = expenses_pivot_l1.loc[:, f"amount ({CURR})"]

fig_l1 = get_sunburst_figure_from_pivot(expenses_pivot_l1_curr, column_to_pick="all_periods")

fig_l1.update_layout(
    margin=dict(t=30, l=0, r=0, b=0),
    width=600,
    height=600,
    title_text="Expenses — Top-level categories only",
    title_font=dict(size=20),
    font=dict(size=12),
)

fig_l1.show()

## Sunburst for a specific time period

You can also filter the data to a specific time period. Here we show expenses for a single year.

In [14]:
# Pivot by year (column="year") to get separate columns per year
expenses_df_yearly = expenses_df.copy()
expenses_df_yearly["year"] = pd.to_datetime(expenses_df_yearly["date"]).dt.year

expenses_pivot_yearly = get_bean_pivot(expenses_df_yearly,
                                        column="year",
                                        max_row_levels=100,
                                        repeat_row_labels=False)

expenses_pivot_yearly_curr = expenses_pivot_yearly.loc[:, f"amount ({CURR})"]
expenses_pivot_yearly_curr

year                              2023     2024
acc_L1        acc_L2                           
Entertainment Books              45.00     0.00
              Movies             30.00    35.00
Food          Coffee             60.00    75.00
              Groceries         400.00   450.00
              Restaurant        120.00   150.00
Health        Doctor            200.00   250.00
Housing       Rent            1,500.00 1,500.00
              Utilities         180.00   200.00
Transport     Fuel               90.00   110.00
              PublicTransport    50.00     0.00

In [15]:
# Pick a single year for the sunburst
YEAR_TO_SHOW = 2023

fig_year = get_sunburst_figure_from_pivot(expenses_pivot_yearly_curr, column_to_pick=YEAR_TO_SHOW)

fig_year.update_layout(
    margin=dict(t=30, l=0, r=0, b=0),
    width=700,
    height=700,
    title_text=f"Expenses in {YEAR_TO_SHOW}",
    title_font=dict(size=20),
    font=dict(size=12),
)

fig_year.show()

## Summary

The typical workflow for creating a sunburst diagram from beancount data is:

1. **Query** postings using `beanquery2df` with a filter (e.g. `account ~ '^Expenses'`) and currency conversion.
2. **Pivot** the result using `get_bean_pivot`, choosing:
   - `max_row_levels` to control depth of hierarchy
   - A `column` for time grouping (or a fixed string like `"all_periods"` to combine everything)
3. **Select** the desired currency column from the pivot (e.g. `df.loc[:, f"amount ({CURR})"]`).
4. **Generate** the figure with `get_sunburst_figure_from_pivot(df, column_to_pick=...)`.
5. **Customize** the Plotly figure layout as needed (title, size, margins, etc.).